In [1]:
import sys; sys.path.append("..")
import numpy as np
from data import get_splits, one_hot
from softmax import softmax, loss_and_grad

d = get_splits()
Xtr, ytr = d["X_train"], d["y_train"]

# Kiểm tra 1: softmax ổn định và mỗi hàng cộng bằng 1
Z = np.array([[1000., 1001., 1002.], [-1000., 0., 1000.]])
print(softmax(Z), softmax(Z).sum(axis=1))      # không có nan, tổng bằng 1

# Kiểm tra 2: với W = 0, mô hình đoán đều 10 lớp, nên loss phải bằng ln(10)
X = Xtr[:1000].astype(np.float64); Y = one_hot(ytr[:1000]).astype(np.float64)
loss, _, _ = loss_and_grad(np.zeros((784, 10)), np.zeros(10), X, Y)
print(loss, np.log(10))                        # cả hai ≈ 2.302585

[[0.09003057 0.24472847 0.66524096]
 [0.         0.         1.        ]] [1. 1.]
2.3025850929940463 2.302585092994046


In [2]:
rng = np.random.default_rng(0)
idx = rng.choice(len(Xtr), 20, replace=False)
X = Xtr[idx].astype(np.float64)          # float64 để sai số số học nhỏ
Y = one_hot(ytr[idx]).astype(np.float64)
W = 0.01 * rng.standard_normal((784, 10))
b = rng.standard_normal(10) * 0.01

def numeric_grad_check(l2, n_w=300, eps=1e-5):
    _, dW, db = loss_and_grad(W, b, X, Y, l2)
    abs_errs, rel_errs = [], []
    # kiểm tra n_w phần tử ngẫu nhiên của W
    for _ in range(n_w):
        i, j = rng.integers(784), rng.integers(10)
        old = W[i, j]
        W[i, j] = old + eps; lp = loss_and_grad(W, b, X, Y, l2)[0]
        W[i, j] = old - eps; lm = loss_and_grad(W, b, X, Y, l2)[0]
        W[i, j] = old
        num = (lp - lm) / (2 * eps)
        abs_errs.append(abs(num - dW[i, j]))
        if abs(num) + abs(dW[i, j]) > 1e-6:
            rel_errs.append(abs(num - dW[i, j]) / (abs(num) + abs(dW[i, j])))
    # kiểm tra toàn bộ 10 phần tử của b
    for j in range(10):
        old = b[j]
        b[j] = old + eps; lp = loss_and_grad(W, b, X, Y, l2)[0]
        b[j] = old - eps; lm = loss_and_grad(W, b, X, Y, l2)[0]
        b[j] = old
        num = (lp - lm) / (2 * eps)
        abs_errs.append(abs(num - db[j]))
        if abs(num) + abs(db[j]) > 1e-6:
            rel_errs.append(abs(num - db[j]) / (abs(num) + abs(db[j])))
    print(f"l2={l2}: sai số tuyệt đối tối đa {max(abs_errs):.2e} | sai số tương đối tối đa {max(rel_errs):.2e}")

numeric_grad_check(l2=0.0)
numeric_grad_check(l2=0.1)     # kiểm tra cả thành phần regularization

l2=0.0: sai số tuyệt đối tối đa 4.16e-11 | sai số tương đối tối đa 1.22e-07
l2=0.1: sai số tuyệt đối tối đa 5.23e-11 | sai số tương đối tối đa 3.43e-07


In [3]:
X = Xtr[:2000].astype(np.float64); Y = one_hot(ytr[:2000]).astype(np.float64)
W = np.zeros((784, 10)); b = np.zeros(10)
for step in range(200):
    loss, dW, db = loss_and_grad(W, b, X, Y)
    W -= 0.5 * dW; b -= 0.5 * db
    if step % 40 == 0: print(step, round(loss, 4))

0 2.3026
40 0.4486
80 0.3526
120 0.305
160 0.2736
